In [1]:
import cornac
from cornac.data import Reader
from cornac.datasets import movielens
from cornac.data import Dataset, FeatureModality
from cornac.eval_methods import RatioSplit, StratifiedSplit
from cornac.metrics import RMSE, AUC, NDCG, Precision, Recall
from cornac.models import MF, ItemKNN, UserKNN, NMF, BPR, LightGCN, SVD, MostPop, VAECF,NeuMF
import pandas as pd
import numpy as np
import random
import math
from collections import OrderedDict
import seaborn as sns
import matplotlib.pyplot as plt

/Users/tahsinalamgirkheya/anaconda3/envs/cornac/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
reader = Reader()
rating_data_pd = pd.read_csv(
    "./cornac/data_c/ml-100k/indexed_interactions.csv",
    sep="\t",
    header=None,
    names=["userID", "itemID", "Rating", "Timestamp"],
)
# print(rating_data["itemID"].nunique())
# rating_data = rating_data.drop(columns=rating_data.columns[-1])
rating_data = rating_data_pd.to_numpy()
rating_data.__len__()
rating_data_pd

,userID,itemID,Rating,Timestamp
0,0,0,3,881250949
1,1,1,3,891717742
2,2,2,1,878887116
3,3,3,2,880606923
4,4,4,1,886397596
...,...,...,...,...
99282,875,173,3,880175444
99283,708,247,5,879795543
99284,37,982,1,874795795
99285,58,442,2,882399156


In [3]:
df_m = pd.read_csv(
    "./cornac/data_c/ml-100K/u.item",
    sep="|",
    names=[
        "movieID",
        "Name",
        "Date",
        "Video_Date",
        "IMDB_URL",
        "unknown",
        "Action",
        "Adventure",
        "Animation",
        "Children's",
        "Comedy",
        "Crime",
        "Documentary",
        "Drama",
        "Fantasy",
        "Film-Noir",
        "Horror",
        "Musical",
        "Mystery",
        "Romance",
        "Sci-Fi",
        "Thriller",
        "War",
        "Western",
    ],
    header=None,
    encoding="latin-1",
)
print(df_m.shape)
df_m = df_m[
    [
        "movieID",
        "Action",
        "Adventure",
        "Animation",
        "Children's",
        "Comedy",
        "Crime",
        "Documentary",
        "Drama",
        "Fantasy",
        "Film-Noir",
        "Horror",
        "Musical",
        "Mystery",
        "Romance",
        "Sci-Fi",
        "Thriller",
        "War",
        "Western",
    ]
]

df_movies_mapped = pd.read_csv(
    "./cornac/data_c/ml-100K/i_id_mapping.csv",
    sep="\t",
    names=["movieID", "itemID"],
    header=None,
    encoding="latin-1",
)
movies = pd.merge(df_m, df_movies_mapped, how="inner", on="movieID")
movies

(1682, 24)


,movieID,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,itemID
0,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,24
1,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,147
2,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,233
3,4,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,47
4,5,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1344,1592,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1305
1345,1597,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1324
1346,1598,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1319
1347,1615,1,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1341


In [4]:
movies = movies.drop(columns=["movieID"])

In [5]:
movies = movies.sort_values(by="itemID")

In [6]:
movies

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,itemID
240,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
300,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1
375,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2
50,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,1,3
344,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1248,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1344
1192,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1345
1176,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1346
1261,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1347


In [7]:

unique_genres = [
    "Action",
    "Thriller",
    "Romance",
    "Western",
    "Children's",
    "Mystery",
    "Fantasy",
    "Film-Noir",
    "Documentary",
    "Comedy",
    "Adventure",
    "Sci-Fi",
    "Horror",
    "Crime",
    "Musical",
    "War",
    "Animation",
    "Drama",
]
genre = movies[unique_genres]
item_features_numpy = genre.to_numpy()

users = pd.read_csv("./cornac/data_c/ml-100k/u_id_mapping.csv", sep="\t")
users
users = users.drop(columns=users.columns[0])
gender_map = {"M": 0, "F": 1}
users["Gender"] = users["Gender"].map(gender_map)
user_features_numpy = users.to_numpy()

In [8]:
movies

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,itemID
240,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
300,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1
375,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2
50,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,1,3
344,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1248,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1344
1192,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1345
1176,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1346
1261,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1347


In [9]:
def create_genre_column(r):
    all_genres = [g for g in unique_genres if r[g] == 1]
    return "|".join(all_genres)


movies["genres"] = movies.apply(create_genre_column, axis=1)
movies

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,itemID,genres
240,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Comedy
300,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,Thriller|Mystery|Film-Noir|Crime
375,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2,Children's|Comedy
50,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,1,3,Romance|Western|War|Drama
344,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,4,Crime|Drama
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1248,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1344,Drama
1192,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1345,Comedy
1176,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1346,Drama
1261,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1347,Drama


In [10]:

dataset = rating_data
unique_genres.__len__()

18

In [11]:
# new_df = inter_movies.groupby("userID")[unique_genres].sum()
# new_df = pd.merge(new_df, users, on="userID", how="inner")
# new_df = inter_movies.groupby("Gender")[unique_genres].sum()
# new_df
users.iloc[714]

Gender      0
userID    714
Name: 714, dtype: int64

In [12]:
# genre_to_remove_m=["Romance", "Drama","Comedy", "Musical"]
# genre_to_remove_f=["Action", "War","Thriller", "Horror"]
# # upper_bound = inter_movies.shape[0]
# indices_to_delete = set()
# mask_f = (inter_movies['Gender'] == 1) & (inter_movies[genre_to_remove_f].sum(axis=1) > 0)
# mask_m = (inter_movies['Gender'] == 0) & (inter_movies[genre_to_remove_m].sum(axis=1) > 0)
# mask_f = inter_movies.index[mask_f]
# mask_m = inter_movies.index[mask_m]
# upper_bound_m = mask_m.__len__()
# upper_bound_f = mask_f.__len__()
# for i in range(100000):
#     rm=random.randint(0, upper_bound_m-1)
#     rf=random.randint(0, upper_bound_f-1)
#     indices_to_delete.add(mask_f[rf])
#     indices_to_delete.add(mask_m[rm])

# inter_movies = inter_movies.drop(indices_to_delete, axis=0).reset_index(drop=True)
# # sales.drop(sales[sales.CustomerId.isin(badcu)].index.tolist()

In [13]:
# inter_movies
# unique_iids = inter_movies['userID'].unique()
# unique_iids.__len__()
# # movies = movies[movies['itemID'].isin(unique_iids)]
user_features_numpy[:, 0]
# item_features_numpy[0]

array([0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
       0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0,
       0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0,

In [14]:
users

,Gender,userID
0,0,0
1,1,1
2,0,2
3,0,3
4,0,4
...,...,...
938,1,938
939,0,939
940,1,940
941,1,941


In [18]:
all_interaction = pd.merge(rating_data_pd, users, on= "userID", how="inner")
x = all_interaction.groupby('userID')["Rating"].count().sort_values(ascending=False)

# Convert the Series to a DataFrame and rename the column
x = x.reset_index(name='interactions')
x = pd.merge(x, users, on = "userID", how = "inner")
# Display the resulting DataFrame with 'userID' and 'interactions'
x_female = x[x["Gender"]==1]
x_male = x[x["Gender"]==0]

x_male_active = x_male[:25]
x_female_active = x_female[:25]



In [19]:
m = x_male_active["userID"].to_numpy()
f =x_female_active["userID"].to_numpy()
f

array([402, 650, 442, 409, 413, 790, 100, 633, 129, 453, 114, 802, 384,
       708, 434, 205, 499, 264, 793, 348, 696, 364,  10, 341, 576])

In [17]:
ratio_split = StratifiedSplit(
    data=dataset,
    test_size=0.2,
    rating_threshold=0,
    val_size=0.1,
    seed=123,
    verbose=True,
    chrono=True,
    user_features=user_features_numpy[:, 0],
    item_features=item_features_numpy,
    exclude_unknowns=False,
)
rec_50 = cornac.metrics.Recall(k=50)
ndcg_50 = cornac.metrics.NDCG(k=50)
auc = cornac.metrics.AUC()
rmse = cornac.metrics.RMSE()
prec = cornac.metrics.Precision(k=50)
hr = cornac.metrics.HitRatio(k=50)
mrr = cornac.metrics.MRR()
map = cornac.metrics.MAP()
f1 = cornac.metrics.FMeasure(k=50)

models = []
model_1 = UserKNN(k=20, seed=123, verbose=True)
model_2 = ItemKNN(k=20, seed=123, verbose=True)
model_3 = MF(
            k=20,
            seed=123,
            name=f"a={0} mf",
            backend="pytorch",
            verbose=True,
            optimizer="adam",batch_size=256,
            alpha=0,
            learning_rate=0.001,
            top_k=50, max_iter=20,
            # early_stopping=True
        )
model_4 = VAECF(k=32,
        autoencoder_structure=[60,40],
        act_fn="relu",
        likelihood="bern",
        n_epochs=50,
        batch_size=128,
        learning_rate=0.0005,
        alpha=0,
        top_k=50,
        beta=1,
        name=f"a={0} vae",
        seed=123,
        verbose=True,
        # early_stopping=True)
)
# model_5 = NeuMF(num_factors=8, layers=[32,16,8], act_fn="relu", num_epochs=64, batch_size=256, num_neg=3, backend="pytorch", lr=0.001, alp=0, top_k=50, name=str(0)+ "neumf")
# models.append(model_5)

models.append(model_1)
models.append(model_2)
models.append(model_3)
models.append(model_4)


cornac.Experiment(
    ratio_split, models=models, metrics=[rec_50]
).run()




rating_threshold = 0.0
exclude_unknowns = False
::::::::
OrderedDict()
OrderedDict()
::::::::
---
Training data:
(array([  0,   0,   0, ..., 942, 942, 942]), array([  0,   1,   2, ..., 880, 632, 261]), array([4., 4., 4., ..., 1., 5., 4.]))
Number of users = 943
Number of items = 1348
Number of ratings = 68674
Max rating = 5.0
Min rating = 1.0
Global mean = 3.6
Global mean Imolicit= 1.0
::::::::
OrderedDict([(101, 0), (845, 1), (705, 2), (20, 3), (635, 4), (593, 5), (153, 6), (37, 7), (915, 8), (522, 9), (810, 10), (18, 11), (34, 12), (809, 13), (239, 14), (749, 15), (889, 16), (928, 17), (542, 18), (501, 19), (342, 20), (175, 21), (116, 22), (144, 23), (618, 24), (118, 25), (860, 26), (335, 27), (263, 28), (825, 29), (405, 30), (942, 31), (867, 32), (286, 33), (393, 34), (224, 35), (146, 36), (792, 37), (416, 38), (652, 39), (147, 40), (368, 41), (773, 42), (727, 43), (194, 44), (246, 45), (75, 46), (446, 47), (76, 48), (461, 49), (793, 50), (912, 51), (110, 52), (504, 53), (84, 54), (

100%|██████████| 943/943 [00:00<00:00, 29914.68it/s]



[UserKNN] Evaluation started!


Ranking: 100%|██████████| 943/943 [00:02<00:00, 366.55it/s]


**********
##########
malemalemalemalemalemalemalemalemalemale
[0, 1, 4, 8, 10, 12, 14, 16, 17, 19, 25, 28, 30, 32, 35, 37, 38, 39, 44, 47, 49, 51, 52, 53, 54, 56, 62, 70, 75, 77, 79, 81, 83, 87, 88, 89, 90, 95, 101, 102, 107, 114, 115, 118, 120, 125, 126, 129, 131, 134, 143, 144, 149, 152, 155, 162, 164, 170, 172, 176, 178, 180, 181, 187, 189, 191, 201, 205, 206, 207, 208, 210, 212, 215, 221, 222, 232, 235, 236, 237, 240, 247, 249, 251, 253, 263, 266, 275, 276, 280, 285, 297, 305, 306, 314, 316, 331, 332, 339, 340, 342, 345, 348, 349, 353, 355, 356, 361, 363, 367, 368, 370, 371, 372, 376, 378, 384, 385, 388, 392, 394, 399, 404, 408, 409, 418, 425, 426, 428, 438, 439, 442, 448, 460, 461, 462, 466, 469, 473, 477, 480, 485, 486, 490, 498, 500, 507, 508, 510, 518, 524, 527, 532, 534, 535, 538, 547, 548, 549, 550, 551, 554, 556, 558, 559, 565, 566, 574, 575, 576, 579, 587, 588, 590, 596, 598, 604, 609, 611, 613, 614, 616, 630, 634, 635, 636, 638, 639, 640, 642, 646, 650, 656, 657, 662, 663

Ranking:  61%|██████    | 577/943 [00:01<00:00, 382.87it/s]


KeyboardInterrupt: 

In [18]:
user_ids = users.to_numpy()[:, 1]
item_ids = movies.to_numpy()[:, 2]


In [19]:
male_uids = [7, 46, 75, 167, 214, 296, 370, 384, 414, 452, 474, 548, 554, 559, 633, 638, 641, 678, 688, 690, 706, 719, 741, 797, 859]
25
female_uids = [50, 127, 146, 179, 243, 264, 303, 322, 338, 354, 365, 377, 381, 470, 476, 595, 618, 620, 682, 694, 770, 817, 882, 911, 930]

In [20]:
all_mapped_users = OrderedDict([(101, 0), (845, 1), (705, 2), (20, 3), (635, 4), (593, 5), (153, 6), (37, 7), (915, 8), (522, 9), (810, 10), (18, 11), (34, 12), (809, 13), (239, 14), (749, 15), (889, 16), (928, 17), (542, 18), (501, 19), (342, 20), (175, 21), (116, 22), (144, 23), (618, 24), (118, 25), (860, 26), (335, 27), (263, 28), (825, 29), (405, 30), (942, 31), (867, 32), (286, 33), (393, 34), (224, 35), (146, 36), (792, 37), (416, 38), (652, 39), (147, 40), (368, 41), (773, 42), (727, 43), (194, 44), (246, 45), (75, 46), (446, 47), (76, 48), (461, 49), (793, 50), (912, 51), (110, 52), (504, 53), (84, 54), (627, 55), (621, 56), (42, 57), (594, 58), (728, 59), (301, 60), (214, 61), (668, 62), (937, 63), (484, 64), (274, 65), (685, 66), (526, 67), (776, 68), (121, 69), (139, 70), (108, 71), (679, 72), (207, 73), (111, 74), (65, 75), (587, 76), (143, 77), (322, 78), (392, 79), (770, 80), (299, 81), (838, 82), (690, 83), (630, 84), (27, 85), (829, 86), (11, 87), (676, 88), (26, 89), (380, 90), (490, 91), (378, 92), (706, 93), (909, 94), (781, 95), (98, 96), (463, 97), (772, 98), (795, 99), (662, 100), (883, 101), (12, 102), (487, 103), (721, 104), (703, 105), (771, 106), (448, 107), (551, 108), (759, 109), (767, 110), (397, 111), (95, 112), (596, 113), (339, 114), (198, 115), (192, 116), (869, 117), (219, 118), (658, 119), (645, 120), (49, 121), (878, 122), (315, 123), (683, 124), (361, 125), (639, 126), (409, 127), (738, 128), (935, 129), (28, 130), (289, 131), (691, 132), (512, 133), (882, 134), (367, 135), (407, 136), (681, 137), (158, 138), (545, 139), (137, 140), (107, 141), (125, 142), (852, 143), (29, 144), (537, 145), (364, 146), (864, 147), (255, 148), (895, 149), (178, 150), (713, 151), (161, 152), (317, 153), (862, 154), (432, 155), (711, 156), (278, 157), (701, 158), (334, 159), (834, 160), (465, 161), (391, 162), (363, 163), (149, 164), (220, 165), (56, 166), (70, 167), (22, 168), (813, 169), (74, 170), (602, 171), (383, 172), (131, 173), (329, 174), (535, 175), (133, 176), (905, 177), (436, 178), (264, 179), (39, 180), (578, 181), (800, 182), (689, 183), (477, 184), (82, 185), (574, 186), (472, 187), (345, 188), (92, 189), (469, 190), (922, 191), (814, 192), (450, 193), (808, 194), (729, 195), (754, 196), (417, 197), (300, 198), (222, 199), (491, 200), (636, 201), (841, 202), (742, 203), (649, 204), (440, 205), (103, 206), (640, 207), (694, 208), (2, 209), (758, 210), (223, 211), (485, 212), (356, 213), (35, 214), (71, 215), (445, 216), (244, 217), (59, 218), (1, 219), (188, 220), (270, 221), (156, 222), (920, 223), (473, 224), (530, 225), (24, 226), (567, 227), (186, 228), (549, 229), (801, 230), (16, 231), (215, 232), (856, 233), (272, 234), (647, 235), (73, 236), (850, 237), (932, 238), (599, 239), (182, 240), (665, 241), (106, 242), (10, 243), (89, 244), (913, 245), (588, 246), (870, 247), (389, 248), (225, 249), (365, 250), (904, 251), (418, 252), (273, 253), (560, 254), (152, 255), (302, 256), (332, 257), (903, 258), (498, 259), (787, 260), (377, 261), (295, 262), (585, 263), (696, 264), (805, 265), (241, 266), (419, 267), (358, 268), (797, 269), (243, 270), (430, 271), (249, 272), (726, 273), (835, 274), (404, 275), (746, 276), (677, 277), (611, 278), (794, 279), (573, 280), (468, 281), (756, 282), (102, 283), (126, 284), (743, 285), (128, 286), (120, 287), (527, 288), (628, 289), (849, 290), (476, 291), (901, 292), (609, 293), (159, 294), (346, 295), (15, 296), (349, 297), (354, 298), (138, 299), (17, 300), (737, 301), (764, 302), (129, 303), (227, 304), (622, 305), (460, 306), (940, 307), (735, 308), (321, 309), (31, 310), (488, 311), (261, 312), (684, 313), (710, 314), (457, 315), (320, 316), (45, 317), (531, 318), (213, 319), (789, 320), (582, 321), (413, 322), (57, 323), (386, 324), (846, 325), (423, 326), (414, 327), (906, 328), (260, 329), (898, 330), (44, 331), (401, 332), (443, 333), (680, 334), (540, 335), (96, 336), (620, 337), (708, 338), (566, 339), (281, 340), (150, 341), (851, 342), (702, 343), (78, 344), (881, 345), (435, 346), (478, 347), (310, 348), (54, 349), (240, 350), (833, 351), (892, 352), (539, 353), (384, 354), (548, 355), (212, 356), (831, 357), (441, 358), (304, 359), (572, 360), (204, 361), (91, 362), (408, 363), (482, 364), (802, 365), (555, 366), (64, 367), (894, 368), (53, 369), (369, 370), (798, 371), (122, 372), (865, 373), (626, 374), (736, 375), (185, 376), (114, 377), (173, 378), (151, 379), (661, 380), (434, 381), (863, 382), (899, 383), (875, 384), (885, 385), (81, 386), (938, 387), (127, 388), (731, 389), (783, 390), (766, 391), (352, 392), (918, 393), (366, 394), (603, 395), (284, 396), (174, 397), (569, 398), (32, 399), (202, 400), (559, 401), (233, 402), (303, 403), (740, 404), (557, 405), (374, 406), (791, 407), (181, 408), (663, 409), (327, 410), (429, 411), (3, 412), (763, 413), (752, 414), (815, 415), (624, 416), (914, 417), (907, 418), (283, 419), (425, 420), (876, 421), (782, 422), (390, 423), (237, 424), (130, 425), (919, 426), (591, 427), (570, 428), (200, 429), (722, 430), (733, 431), (518, 432), (817, 433), (406, 434), (55, 435), (6, 436), (755, 437), (682, 438), (257, 439), (197, 440), (0, 441), (183, 442), (590, 443), (866, 444), (732, 445), (141, 446), (256, 447), (454, 448), (890, 449), (187, 450), (253, 451), (58, 452), (666, 453), (282, 454), (558, 455), (343, 456), (564, 457), (493, 458), (77, 459), (447, 460), (166, 461), (704, 462), (30, 463), (553, 464), (117, 465), (550, 466), (135, 467), (816, 468), (644, 469), (442, 470), (532, 471), (394, 472), (360, 473), (421, 474), (803, 475), (453, 476), (888, 477), (565, 478), (525, 479), (672, 480), (762, 481), (753, 482), (617, 483), (583, 484), (687, 485), (873, 486), (265, 487), (847, 488), (235, 489), (47, 490), (779, 491), (444, 492), (634, 493), (924, 494), (458, 495), (123, 496), (580, 497), (494, 498), (228, 499), (848, 500), (242, 501), (199, 502), (529, 503), (9, 504), (60, 505), (517, 506), (716, 507), (592, 508), (614, 509), (470, 510), (437, 511), (290, 512), (552, 513), (692, 514), (891, 515), (861, 516), (479, 517), (99, 518), (279, 519), (502, 520), (251, 521), (221, 522), (464, 523), (589, 524), (298, 525), (513, 526), (113, 527), (112, 528), (288, 529), (619, 530), (201, 531), (61, 532), (511, 533), (497, 534), (720, 535), (79, 536), (839, 537), (719, 538), (399, 539), (505, 540), (523, 541), (259, 542), (607, 543), (536, 544), (496, 545), (604, 546), (218, 547), (837, 548), (822, 549), (210, 550), (579, 551), (136, 552), (160, 553), (41, 554), (896, 555), (584, 556), (5, 557), (483, 558), (428, 559), (686, 560), (154, 561), (85, 562), (941, 563), (659, 564), (33, 565), (86, 566), (516, 567), (351, 568), (660, 569), (88, 570), (675, 571), (734, 572), (786, 573), (921, 574), (226, 575), (69, 576), (191, 577), (784, 578), (285, 579), (929, 580), (519, 581), (877, 582), (697, 583), (231, 584), (238, 585), (933, 586), (715, 587), (395, 588), (698, 589), (157, 590), (309, 591), (547, 592), (931, 593), (340, 594), (341, 595), (43, 596), (268, 597), (900, 598), (336, 599), (163, 600), (449, 601), (355, 602), (313, 603), (509, 604), (744, 605), (544, 606), (247, 607), (520, 608), (673, 609), (40, 610), (459, 611), (709, 612), (853, 613), (812, 614), (109, 615), (761, 616), (375, 617), (402, 618), (275, 619), (633, 620), (632, 621), (403, 622), (46, 623), (533, 624), (820, 625), (306, 626), (884, 627), (826, 628), (712, 629), (575, 630), (855, 631), (168, 632), (556, 633), (481, 634), (563, 635), (83, 636), (51, 637), (90, 638), (422, 639), (615, 640), (534, 641), (252, 642), (521, 643), (93, 644), (745, 645), (48, 646), (8, 647), (456, 648), (688, 649), (206, 650), (818, 651), (455, 652), (4, 653), (717, 654), (97, 655), (382, 656), (373, 657), (67, 658), (379, 659), (347, 660), (538, 661), (887, 662), (372, 663), (638, 664), (433, 665), (741, 666), (939, 667), (610, 668), (601, 669), (293, 670), (568, 671), (87, 672), (842, 673), (524, 674), (828, 675), (910, 676), (748, 677), (893, 678), (595, 679), (66, 680), (314, 681), (650, 682), (480, 683), (796, 684), (507, 685), (515, 686), (13, 687), (23, 688), (311, 689), (387, 690), (879, 691), (671, 692), (431, 693), (499, 694), (203, 695), (648, 696), (467, 697), (318, 698), (858, 699), (105, 700), (14, 701), (169, 702), (262, 703), (236, 704), (868, 705), (52, 706), (832, 707), (230, 708), (902, 709), (297, 710), (936, 711), (258, 712), (167, 713), (750, 714), (598, 715), (629, 716), (871, 717), (211, 718), (674, 719), (656, 720), (765, 721), (229, 722), (769, 723), (508, 724), (277, 725), (193, 726), (411, 727), (500, 728), (489, 729), (886, 730), (597, 731), (739, 732), (172, 733), (605, 734), (232, 735), (561, 736), (667, 737), (699, 738), (370, 739), (707, 740), (68, 741), (785, 742), (305, 743), (854, 744), (208, 745), (641, 746), (145, 747), (184, 748), (276, 749), (269, 750), (859, 751), (148, 752), (307, 753), (471, 754), (196, 755), (426, 756), (381, 757), (700, 758), (287, 759), (657, 760), (195, 761), (410, 762), (362, 763), (466, 764), (725, 765), (162, 766), (503, 767), (613, 768), (506, 769), (576, 770), (179, 771), (451, 772), (462, 773), (250, 774), (806, 775), (291, 776), (164, 777), (190, 778), (50, 779), (777, 780), (94, 781), (338, 782), (836, 783), (439, 784), (248, 785), (747, 786), (730, 787), (916, 788), (581, 789), (897, 790), (396, 791), (333, 792), (623, 793), (824, 794), (799, 795), (245, 796), (19, 797), (823, 798), (554, 799), (843, 800), (788, 801), (664, 802), (216, 803), (412, 804), (475, 805), (844, 806), (923, 807), (807, 808), (292, 809), (562, 810), (388, 811), (528, 812), (543, 813), (324, 814), (655, 815), (124, 816), (348, 817), (669, 818), (495, 819), (927, 820), (385, 821), (654, 822), (830, 823), (171, 824), (474, 825), (751, 826), (840, 827), (323, 828), (872, 829), (612, 830), (616, 831), (631, 832), (925, 833), (625, 834), (723, 835), (217, 836), (331, 837), (350, 838), (209, 839), (398, 840), (328, 841), (571, 842), (718, 843), (586, 844), (934, 845), (424, 846), (541, 847), (514, 848), (492, 849), (344, 850), (36, 851), (371, 852), (180, 853), (359, 854), (768, 855), (608, 856), (115, 857), (930, 858), (38, 859), (821, 860), (415, 861), (486, 862), (775, 863), (646, 864), (142, 865), (80, 866), (780, 867), (104, 868), (874, 869), (420, 870), (695, 871), (757, 872), (326, 873), (400, 874), (357, 875), (176, 876), (165, 877), (132, 878), (651, 879), (917, 880), (637, 881), (100, 882), (337, 883), (693, 884), (642, 885), (271, 886), (510, 887), (234, 888), (819, 889), (760, 890), (312, 891), (308, 892), (857, 893), (353, 894), (778, 895), (140, 896), (316, 897), (325, 898), (438, 899), (62, 900), (170, 901), (330, 902), (7, 903), (774, 904), (134, 905), (606, 906), (319, 907), (189, 908), (880, 909), (254, 910), (205, 911), (827, 912), (177, 913), (811, 914), (452, 915), (72, 916), (926, 917), (600, 918), (155, 919), (911, 920), (267, 921), (643, 922), (266, 923), (427, 924), (296, 925), (653, 926), (280, 927), (25, 928), (804, 929), (790, 930), (119, 931), (670, 932), (376, 933), (546, 934), (21, 935), (714, 936), (908, 937), (577, 938), (294, 939), (63, 940), (678, 941), (724, 942)])


In [21]:
# get the top_k ratings for all users:
top_k = 10
reco_matrix = np.zeros((len(models), len(user_ids), top_k), dtype=int)
reco_matrix_mapped_items = np.zeros(
    (len(models), len(user_ids), len(item_ids)), dtype=int
)
reco_matrix_mapped_scores = np.zeros(
    (len(models), len(user_ids), len(item_ids)), dtype=float
)
reco_matrix_all = np.zeros((len(models), len(user_ids), len(item_ids)), dtype=int)


for u in user_ids:
    for i in range(len(models)):
        reco_items = models[i].recommend(u)
        items_mapped, mapped_scores = models[i].rank(
            user_idx=u, item_indices=list(item_ids)
        )
        reco_matrix_mapped_items[i][u] = items_mapped
        reco_matrix_mapped_scores[i][u] = mapped_scores
        reco_matrix_all[i][u] = reco_items
        reco_matrix[i][u] = reco_items[:top_k]

        # print(reco_matrix[0][3])

In [22]:
male_uids_new=[]
for i in male_uids:
    for k, v in all_mapped_users.items():
        if v==i:
            # print(f"actual id {k} mapped {v}")
            male_uids_new.append(k)
female_uids_new=[]
for i in female_uids:
    for k, v in all_mapped_users.items():
        if v==i:
            female_uids_new.append(k)
print(len(male_uids_new))
print(len(female_uids_new))
for i in range(len(models)):
    df_reco = pd.DataFrame(
            {
                "userID": np.repeat(np.arange(reco_matrix[i].shape[0]), top_k),
                "itemID": reco_matrix[i].flatten(),
                "rank": np.tile(np.arange(1, top_k + 1), reco_matrix[i].shape[0]),
            }
        )

    
    all_ids = male_uids_new+female_uids_new
    df_reco = df_reco[df_reco['userID'].isin(all_ids)]

    df_reco = pd.merge(df_reco, movies, on= "itemID", how="inner")
    df_reco_new = df_reco
    # df_reco_new = df_reco_new.groupby
    df_reco = df_reco.groupby("userID")[unique_genres].sum()
    x_df = df_reco[unique_genres]/10
    
    df_reco = pd.merge(df_reco, users, on= "userID", how="inner")
    x_df = pd.merge(x_df, users, on= "userID", how="inner")
    x_df = x_df.groupby("Gender")[unique_genres].sum()

    # df_reco = df_reco.groupby("Gender")[unique_genres].sum()
    # print("::::::::")
    # print(df_reco[["Action","Romance"]])
    # # print(df_reco)
    # print(models[i].name)
    # print("::::::::")
    
    #  df_reco = df_reco.groupby("Gender")[unique_genres].sum()
    print("::::::::")
    print(x_df[["Action","Romance"]])
    # print(df_reco)
    print(models[i].name)
    print("::::::::")
    



25
25
::::::::
        Action  Romance
Gender                 
0          5.6      6.3
1          4.6      5.7
UserKNN
::::::::
::::::::
        Action  Romance
Gender                 
0          5.4      4.3
1          2.7      5.3
ItemKNN
::::::::
::::::::
        Action  Romance
Gender                 
0          7.2      5.0
1          7.2      6.6
a=0 mf
::::::::
::::::::
        Action  Romance
Gender                 
0         16.7      5.8
1         16.1      6.1
a=0 vae
::::::::


In [23]:
df_reco

,userID,Action,Thriller,Romance,Western,Children's,Mystery,Fantasy,Film-Noir,Documentary,Comedy,Adventure,Sci-Fi,Horror,Crime,Musical,War,Animation,Drama,Gender
0,10,5,3,3,0,0,0,0,0,0,1,3,6,1,1,0,3,0,5,1
1,15,7,5,2,0,1,0,1,0,0,1,3,7,1,1,0,3,0,3,0
2,19,6,4,3,0,1,0,1,0,0,2,3,6,0,1,0,4,0,3,0
3,23,7,5,2,0,1,0,1,0,0,1,3,7,1,1,0,3,0,3,0
4,35,6,4,3,0,1,0,1,0,0,2,3,6,0,1,0,4,0,3,0
5,37,6,4,3,0,1,0,1,0,0,2,3,6,0,1,0,4,0,3,0
6,38,7,5,2,0,1,0,1,0,0,1,3,7,1,1,0,3,0,3,0
7,41,7,5,2,0,1,0,1,0,0,1,3,7,1,1,0,3,0,3,0
8,52,7,5,2,0,1,0,1,0,0,1,3,7,1,1,0,3,0,3,0
9,58,6,4,3,0,1,0,1,0,0,2,3,6,0,1,0,4,0,3,0


In [24]:
print(len([2, 6, 15, 23, 26, 27, 42, 45, 50, 64, 65, 66, 73, 74, 76, 78, 80, 82, 86, 91, 96, 103, 105, 109, 113, 117, 123, 127, 128, 133, 135, 136, 138, 146, 148, 150, 151, 156, 158, 161, 166, 179, 195, 199, 211, 217, 219, 227, 228, 230, 231, 234, 242, 243, 245, 248, 254, 257, 258, 259, 262, 264, 265, 270, 277, 282, 287, 292, 301, 303, 304, 307, 308, 310, 311, 312, 318, 322, 323, 324, 325, 328, 329, 330, 334, 335, 336, 338, 341, 346, 351, 354, 359, 360, 362, 365, 369, 374, 377, 381, 383, 387, 389, 397, 398, 400, 401, 402, 403, 405, 410, 419, 420, 421, 427, 432, 440, 444, 449, 453, 464, 465, 467, 468, 470, 476, 479, 481, 482, 488, 491, 492, 494, 495, 496, 502, 512, 513, 514, 515, 516, 521, 522, 525, 528, 529, 531, 533, 536, 537, 543, 544, 546, 553, 560, 561, 562, 563, 564, 568, 571, 577, 586, 589, 592, 594, 595, 597, 602, 612, 618, 619, 620, 621, 622, 626, 627, 631, 632, 644, 645, 648, 649, 651, 655, 660, 664, 665, 668, 674, 676, 677, 680, 682, 683, 691, 692, 693, 694, 695, 696, 700, 701, 703, 716, 717, 720, 722, 725, 727, 728, 731, 739, 740, 742, 753, 759, 765, 767, 768, 770, 771, 772, 779, 782, 787, 789, 791, 793, 794, 800, 803, 808, 809, 812, 817, 819, 831, 832, 835, 837, 839, 843, 845, 846, 848, 850, 851, 856, 858, 869, 871, 875, 880, 882, 884, 893, 895, 896, 897, 900, 903, 905, 911, 916, 917, 924, 925, 927, 928, 929, 930, 937]
))

273


In [25]:
males = [0, 1, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 22, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 46, 47, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 67, 68, 69, 70, 71, 72, 75, 77, 79, 81, 83, 84, 85, 87, 88, 89, 90, 92, 93, 94, 95, 97, 98, 99, 100, 101, 102, 104, 106, 107, 108, 110, 111, 112, 114, 115, 116, 118, 119, 120, 121, 122, 124, 125, 126, 129, 130, 131, 132, 134, 137, 139, 140, 141, 142, 143, 144, 145, 147, 149, 152, 153, 154, 155, 157, 159, 160, 162, 163, 164, 165, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 196, 197, 198, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 212, 213, 214, 215, 216, 218, 220, 221, 222, 223, 224, 225, 226, 229, 232, 233, 235, 236, 237, 238, 239, 240, 241, 244, 246, 247, 249, 250, 251, 252, 253, 255, 256, 260, 261, 263, 266, 267, 268, 269, 271, 272, 273, 274, 275, 276, 278, 279, 280, 281, 283, 284, 285, 286, 288, 289, 290, 291, 293, 294, 295, 296, 297, 298, 299, 300, 302, 305, 306, 309, 313, 314, 315, 316, 317, 319, 320, 321, 326, 327, 331, 332, 333, 337, 339, 340, 342, 343, 344, 345, 347, 348, 349, 350, 352, 353, 355, 356, 357, 358, 361, 363, 364, 366, 367, 368, 370, 371, 372, 373, 375, 376, 378, 379, 380, 382, 384, 385, 386, 388, 390, 391, 392, 393, 394, 395, 396, 399, 404, 406, 407, 408, 409, 411, 412, 413, 414, 415, 416, 417, 418, 422, 423, 424, 425, 426, 428, 429, 430, 431, 433, 434, 435, 436, 437, 438, 439, 441, 442, 443, 445, 446, 447, 448, 450, 451, 452, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 466, 469, 471, 472, 473, 474, 475, 477, 478, 480, 483, 484, 485, 486, 487, 489, 490, 493, 497, 498, 499, 500, 501, 503, 504, 505, 506, 507, 508, 509, 510, 511, 517, 518, 519, 520, 523, 524, 526, 527, 530, 532, 534, 535, 538, 539, 540, 541, 542, 545, 547, 548, 549, 550, 551, 552, 554, 555, 556, 557, 558, 559, 565, 566, 567, 569, 570, 572, 573, 574, 575, 576, 578, 579, 580, 581, 582, 583, 584, 585, 587, 588, 590, 591, 593, 596, 598, 599, 600, 601, 603, 604, 605, 606, 607, 608, 609, 610, 611, 613, 614, 615, 616, 617, 623, 624, 625, 628, 629, 630, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 646, 647, 650, 652, 653, 654, 656, 657, 658, 659, 661, 662, 663, 666, 667, 669, 670, 671, 672, 673, 675, 678, 679, 681, 684, 685, 686, 687, 688, 689, 690, 697, 698, 699, 702, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 718, 719, 721, 723, 724, 726, 729, 730, 732, 733, 734, 735, 736, 737, 738, 741, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 754, 755, 756, 757, 758, 760, 761, 762, 763, 764, 766, 769, 773, 774, 775, 776, 777, 778, 780, 781, 783, 784, 785, 786, 788, 790, 792, 795, 796, 797, 798, 799, 801, 802, 804, 805, 806, 807, 810, 811, 813, 814, 815, 816, 818, 820, 821, 822, 823, 824, 825, 826, 827, 828, 829, 830, 833, 834, 836, 838, 840, 841, 842, 844, 847, 849, 852, 853, 854, 855, 857, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 870, 872, 873, 874, 876, 877, 878, 879, 881, 883, 885, 886, 887, 888, 889, 890, 891, 892, 894, 898, 899, 901, 902, 904, 906, 907, 908, 909, 910, 912, 913, 914, 915, 918, 919, 920, 921, 922, 923, 926, 931, 932, 933, 934, 935, 936, 938, 939, 940, 941, 942]


In [26]:
import numpy as np
np.random.seed(123)
selected_items_males = np.random.choice(males, 273, replace=False)
selected_items_males


array([232, 762,  57, 935, 926,  69, 361, 379, 733, 372,  52, 503, 888,
        13, 891, 599,  59, 411, 734, 429, 208, 104, 736, 273, 343, 534,
       140, 511, 557, 885, 679, 180, 352,  99, 828,  16, 623, 431, 189,
       408, 605, 769, 750, 932, 474, 210, 157, 548, 116, 409, 633, 108,
       746, 766, 865, 783, 218, 558, 729, 798, 100, 274, 873, 203, 392,
       931, 252, 295, 469, 542, 713, 604, 466, 251, 198, 288, 659, 666,
       298, 504,  63, 233,  71, 636, 625, 175, 712, 662, 744, 237,   7,
       418, 370, 938, 193, 721, 738, 263, 214, 173, 786, 841, 615, 606,
       455, 371,  41, 172,  37, 414, 652, 192, 675, 840, 797, 635, 863,
       855,  11, 689, 206, 907, 614, 667, 159, 247, 637, 754, 764, 578,
       425, 590, 309, 337,  48, 131, 921, 240, 524,  84, 423, 224, 477,
        95, 363, 826, 776, 130, 215, 260, 266, 160, 450, 283,  28, 294,
       187, 366, 805, 629, 373, 333, 184, 551,  43, 711, 868, 200,  49,
       608, 327, 795, 497,  79,  40,  39, 350, 790, 820, 155,  1

In [27]:
print(males)

[0, 1, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 22, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 46, 47, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 67, 68, 69, 70, 71, 72, 75, 77, 79, 81, 83, 84, 85, 87, 88, 89, 90, 92, 93, 94, 95, 97, 98, 99, 100, 101, 102, 104, 106, 107, 108, 110, 111, 112, 114, 115, 116, 118, 119, 120, 121, 122, 124, 125, 126, 129, 130, 131, 132, 134, 137, 139, 140, 141, 142, 143, 144, 145, 147, 149, 152, 153, 154, 155, 157, 159, 160, 162, 163, 164, 165, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 196, 197, 198, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 212, 213, 214, 215, 216, 218, 220, 221, 222, 223, 224, 225, 226, 229, 232, 233, 235, 236, 237, 238, 239, 240, 241, 244, 246, 247, 249, 250, 251, 252, 253, 255, 256, 260, 261, 263, 266, 267, 268, 269, 271, 272, 273, 274, 275, 276, 278, 279, 280, 281, 2

In [28]:
len(selected_items_males)

273

In [29]:
m = [0, 1, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 22, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 46, 47, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 67, 68, 69, 70, 71, 72, 75, 77, 79, 81, 83, 84, 85, 87, 88, 89, 90, 92, 93, 94, 95, 97, 98, 99, 100, 101, 102, 104, 106, 107, 108, 110, 111, 112, 114, 115, 116, 118, 119, 120, 121, 122, 124, 125, 126, 129, 130, 131, 132, 134, 137, 139, 140, 141, 142, 143, 144, 145, 147, 149, 152, 153, 154, 155, 157, 159, 160, 162, 163, 164, 165, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 196, 197, 198, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 212, 213, 214, 215, 216, 218, 220, 221, 222, 223, 224, 225, 226, 229, 232, 233, 235, 236, 237, 238, 239, 240, 241, 244, 246, 247, 249, 250, 251, 252, 253, 255, 256, 260, 261, 263, 266, 267, 268, 269, 271, 272, 273, 274, 275, 276, 278, 279, 280, 281, 283, 284, 285, 286, 288, 289, 290, 291, 293, 294, 295, 296, 297, 298, 299, 300, 302, 305, 306, 309, 313, 314, 315, 316, 317, 319, 320, 321, 326, 327, 331, 332, 333, 337, 339, 340, 342, 343, 344, 345, 347, 348, 349, 350, 352, 353, 355, 356, 357, 358, 361, 363, 364, 366, 367, 368, 370, 371, 372, 373, 375, 376, 378, 379, 380, 382, 384, 385, 386, 388, 390, 391, 392, 393, 394, 395, 396, 399, 404, 406, 407, 408, 409, 411, 412, 413, 414, 415, 416, 417, 418, 422, 423, 424, 425, 426, 428, 429, 430, 431, 433, 434, 435, 436, 437, 438, 439, 441, 442, 443, 445, 446, 447, 448, 450, 451, 452, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 466, 469, 471, 472, 473, 474, 475, 477, 478, 480, 483, 484, 485, 486, 487, 489, 490, 493, 497, 498, 499, 500, 501, 503, 504, 505, 506, 507, 508, 509, 510, 511, 517, 518, 519, 520, 523, 524, 526, 527, 530, 532, 534, 535, 538, 539, 540, 541, 542, 545, 547, 548, 549, 550, 551, 552, 554, 555, 556, 557, 558, 559, 565, 566, 567, 569, 570, 572, 573, 574, 575, 576, 578, 579, 580, 581, 582, 583, 584, 585, 587, 588, 590, 591, 593, 596, 598, 599, 600, 601, 603, 604, 605, 606, 607, 608, 609, 610, 611, 613, 614, 615, 616, 617, 623, 624, 625, 628, 629, 630, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 646, 647, 650, 652, 653, 654, 656, 657, 658, 659, 661, 662, 663, 666, 667, 669, 670, 671, 672, 673, 675, 678, 679, 681, 684, 685, 686, 687, 688, 689, 690, 697, 698, 699, 702, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 718, 719, 721, 723, 724, 726, 729, 730, 732, 733, 734, 735, 736, 737, 738, 741, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 754, 755, 756, 757, 758, 760, 761, 762, 763, 764, 766, 769, 773, 774, 775, 776, 777, 778, 780, 781, 783, 784, 785, 786, 788, 790, 792, 795, 796, 797, 798, 799, 801, 802, 804, 805, 806, 807, 810, 811, 813, 814, 815, 816, 818, 820, 821, 822, 823, 824, 825, 826, 827, 828, 829, 830, 833, 834, 836, 838, 840, 841, 842, 844, 847, 849, 852, 853, 854, 855, 857, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 870, 872, 873, 874, 876, 877, 878, 879, 881, 883, 885, 886, 887, 888, 889, 890, 891, 892, 894, 898, 899, 901, 902, 904, 906, 907, 908, 909, 910, 912, 913, 914, 915, 918, 919, 920, 921, 922, 923, 926, 931, 932, 933, 934, 935, 936, 938, 939, 940, 941, 942]


In [30]:
len([0, 1, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 22, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 46, 47, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 67, 68, 69, 70, 71, 72, 75, 77, 79, 81, 83, 84, 85, 87, 88, 89, 90, 92, 93, 94, 95, 97, 98, 99, 100, 101, 102, 104, 106, 107, 108, 110, 111, 112, 114, 115, 116, 118, 119, 120, 121, 122, 124, 125, 126, 129, 130, 131, 132, 134, 137, 139, 140, 141, 142, 143, 144, 145, 147, 149, 152, 153, 154, 155, 157, 159, 160, 162, 163, 164, 165, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 196, 197, 198, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 212, 213, 214, 215, 216, 218, 220, 221, 222, 223, 224, 225, 226, 229, 232, 233, 235, 236, 237, 238, 239, 240, 241, 244, 246, 247, 249, 250, 251, 252, 253, 255, 256, 260, 261, 263, 266, 267, 268, 269, 271, 272, 273, 274, 275, 276, 278, 279, 280, 281, 283, 284, 285, 286, 288, 289, 290, 291, 293, 294, 295, 296, 297, 298, 299, 300, 302, 305, 306, 309, 313, 314, 315, 316, 317, 319, 320, 321, 326, 327, 331, 332, 333, 337, 339, 340, 342, 343, 344, 345, 347, 348, 349, 350, 352, 353, 355, 356, 357, 358, 361, 363, 364, 366, 367, 368, 370, 371, 372, 373, 375, 376, 378, 379, 380, 382, 384, 385, 386, 388, 390, 391, 392, 393, 394, 395, 396, 399, 404, 406, 407, 408, 409, 411, 412, 413, 414, 415, 416, 417, 418, 422, 423, 424, 425, 426, 428, 429, 430, 431, 433, 434, 435, 436, 437, 438, 439, 441, 442, 443, 445, 446, 447, 448, 450, 451, 452, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 466, 469, 471, 472, 473, 474, 475, 477, 478, 480, 483, 484, 485, 486, 487, 489, 490, 493, 497, 498, 499, 500, 501, 503, 504, 505, 506, 507, 508, 509, 510, 511, 517, 518, 519, 520, 523, 524, 526, 527, 530, 532, 534, 535, 538, 539, 540, 541, 542, 545, 547, 548, 549, 550, 551, 552, 554, 555, 556, 557, 558, 559, 565, 566, 567, 569, 570, 572, 573, 574, 575, 576, 578, 579, 580, 581, 582, 583, 584, 585, 587, 588, 590, 591, 593, 596, 598, 599, 600, 601, 603, 604, 605, 606, 607, 608, 609, 610, 611, 613, 614, 615, 616, 617, 623, 624, 625, 628, 629, 630, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 646, 647, 650, 652, 653, 654, 656, 657, 658, 659, 661, 662, 663, 666, 667, 669, 670, 671, 672, 673, 675, 678, 679, 681, 684, 685, 686, 687, 688, 689, 690, 697, 698, 699, 702, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 718, 719, 721, 723, 724, 726, 729, 730, 732, 733, 734, 735, 736, 737, 738, 741, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 754, 755, 756, 757, 758, 760, 761, 762, 763, 764, 766, 769, 773, 774, 775, 776, 777, 778, 780, 781, 783, 784, 785, 786, 788, 790, 792, 795, 796, 797, 798, 799, 801, 802, 804, 805, 806, 807, 810, 811, 813, 814, 815, 816, 818, 820, 821, 822, 823, 824, 825, 826, 827, 828, 829, 830, 833, 834, 836, 838, 840, 841, 842, 844, 847, 849, 852, 853, 854, 855, 857, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 870, 872, 873, 874, 876, 877, 878, 879, 881, 883, 885, 886, 887, 888, 889, 890, 891, 892, 894, 898, 899, 901, 902, 904, 906, 907, 908, 909, 910, 912, 913, 914, 915, 918, 919, 920, 921, 922, 923, 926, 931, 932, 933, 934, 935, 936, 938, 939, 940, 941, 942]
)

670

In [31]:
print(males==m)

True


In [32]:
female_active = [
            402,
            650,
            442,
            409,
            413,
            790,
            100,
            633,
            129,
            453,
            114,
            802,
            384,
            708,
            434,
            205,
            499,
            264,
            793,
            348,
            696,
            364,
            10,
            341,
            576,
        ]
mapped_f=[]
for i in female_active:
    for k, v in all_mapped_users.items():
        if k==i:
            # print(f"actual id {k} mapped {v}")
            mapped_f.append(v)
mapped_f

[618,
 682,
 470,
 127,
 322,
 930,
 882,
 620,
 303,
 476,
 377,
 365,
 354,
 338,
 381,
 911,
 694,
 179,
 50,
 817,
 264,
 146,
 243,
 595,
 770]

In [33]:
male_active=[
            58,
            37,
            534,
            15,
            19,
            387,
            75,
            421,
            837,
            90,
            38,
            674,
            23,
            65,
            68,
            70,
            52,
            41,
            35,
            428,
            369,
            875,
            893,
            556,
            752,
        ]
mapped_m=[]
for i in male_active:
    for k, v in all_mapped_users.items():
        if k==i:
            # print(f"actual id {k} mapped {v}")
            mapped_m.append(v)
mapped_m

[452,
 7,
 641,
 296,
 797,
 690,
 46,
 474,
 548,
 638,
 859,
 719,
 688,
 75,
 741,
 167,
 706,
 554,
 214,
 559,
 370,
 384,
 678,
 633,
 414]

In [34]:
# get the top_k ratings for all users:
top_k = 10
reco_matrix_ = np.zeros((len(models), len(user_ids), top_k), dtype=int)
reco_matrix_mapped_items_ = np.zeros(
    (len(models), len(user_ids), len(item_ids)), dtype=int
)
reco_matrix_mapped_scores_ = np.zeros(
    (len(models), len(user_ids), len(item_ids)), dtype=float
)
reco_matrix_all_ = np.zeros((len(models), len(user_ids), len(item_ids)), dtype=int)


for u in user_ids:
    for i in range(len(models)):
        reco_items = models[i].recommend(u)
        items_mapped, mapped_scores = models[i].rank(
            user_idx=u, item_indices=list(item_ids)
        )
        reco_matrix_mapped_items_[i][u] = items_mapped
        reco_matrix_mapped_scores_[i][u] = mapped_scores
        reco_matrix_all_[i][u] = reco_items
        reco_matrix_[i][u] = reco_items[:top_k]

        # print(reco_matrix[0][3])

KeyboardInterrupt: 